# FedUCA Extra-B: ISIC 2019 Generalizability

Tests whether FedAvg and PGA generalize beyond HAM10000.

**Step 1:** Cache ISIC 2019 ViT embeddings (Cell 1-5)
**Step 2:** Run FedAvg + PGA on ISIC 2019 (Cell 6-8)

**Add data:** `skin-lesion-analysis-toward-melanoma-detection-2019` (ISIC 2019 Kaggle dataset) | **GPU T4** | Internet OFF

Alternative dataset name: `isic-2019`

In [ ]:
import os, json, time, copy, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader, Dataset, TensorDataset
from PIL import Image
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_extra_b"
EMBED_DIR_B = "/kaggle/working/isic2019_embeddings"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EMBED_DIR_B, exist_ok=True)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 8   # ISIC 2019 has 8 classes
EMBED_DIM   = 768
print(f"Device: {DEVICE}")

# ISIC 2019 class names
ISIC19_CLASSES = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']

In [ ]:
# Find ISIC 2019 dataset
print("Searching for ISIC 2019 dataset...")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if 'ISIC_2019' in f and f.endswith('.csv'):
            print(f"  Found: {os.path.join(root, f)}")

# Find ground truth CSV
gt_hits = glob.glob("/kaggle/input/**/ISIC_2019_Training_GroundTruth.csv",
                    recursive=True)
meta_hits = glob.glob("/kaggle/input/**/ISIC_2019_Training_Metadata.csv",
                      recursive=True)

if not gt_hits:
    # Try alternate names
    gt_hits = glob.glob("/kaggle/input/**/*GroundTruth*.csv", recursive=True)
    print("Alternate CSV hits:", gt_hits[:3])

print(f"Ground truth CSV: {gt_hits[0] if gt_hits else 'NOT FOUND'}")

# Sample jpg to confirm image location
jpg_sample = glob.glob("/kaggle/input/**/*.jpg", recursive=True)[:3]
print(f"Sample images: {jpg_sample}")

In [ ]:
# Load ISIC 2019 metadata and build lesion-disjoint split
# FIX A: Use ISIC_2019_Training_Metadata.csv for real lesion_id
# FIX B: Remove HAM10000 overlap (7,470 images by lesion_id matching)

if not gt_hits:
    raise FileNotFoundError(
        "ISIC 2019 ground truth CSV not found. "
        "Add the ISIC 2019 dataset.")

gt = pd.read_csv(gt_hits[0])
print(f"Ground truth shape: {gt.shape}")
class_cols = [c for c in gt.columns if c != 'image']
gt['label'] = gt[class_cols].values.argmax(axis=1)
gt['dx']    = gt[class_cols].idxmax(axis=1)

# FIX A: Load metadata CSV for real lesion identifiers
if meta_hits:
    meta = pd.read_csv(meta_hits[0])
    print(f"Metadata columns: {list(meta.columns)}")
    # ISIC metadata contains 'isic_id', 'patient_id', or 'lesion_id'
    lesion_col = next((c for c in meta.columns
                       if 'lesion' in c.lower() or 'patient' in c.lower()), None)
    if lesion_col:
        gt = gt.merge(meta[['image', lesion_col]].rename(
                          columns={lesion_col: 'lesion_id'}),
                      on='image', how='left')
        print(f"Merged lesion_id from metadata column '{lesion_col}'")
        print(f"Missing lesion_id: {gt['lesion_id'].isna().sum()}")
        # Fill missing with image id as fallback (image-disjoint at minimum)
        gt['lesion_id'] = gt['lesion_id'].fillna(gt['image'])
    else:
        print("WARNING: No lesion/patient column found in metadata. "
              "Using image id as lesion_id (image-disjoint split only).")
        gt['lesion_id'] = gt['image']
else:
    print("WARNING: Metadata CSV not found. Using image id as lesion_id.")
    gt['lesion_id'] = gt['image']

# FIX B: Remove HAM10000 overlapping images by lesion_id matching
# HAM10000 lesion IDs have format HAM_XXXXXXX
ham_lesion_ids = set(
    lid for lid in gt['lesion_id'].dropna()
    if str(lid).startswith('HAM_')
)
print(f"HAM10000-format lesion IDs found in ISIC 2019: {len(ham_lesion_ids)}")
n_before = len(gt)
gt = gt[~gt['lesion_id'].isin(ham_lesion_ids)].reset_index(drop=True)
print(f"Removed {n_before - len(gt)} HAM10000-overlapping images. "
      f"Remaining: {len(gt)}")

print(f"\nClass distribution after deduplication:")
print(gt['dx'].value_counts())
print(f"Total: {len(gt)} images")


In [ ]:
# Lesion-disjoint split for ISIC 2019
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
tv_idx, te_idx = next(gss1.split(gt, gt['label'], groups=gt['lesion_id']))
gt_tv, gt_test = gt.iloc[tv_idx].reset_index(drop=True), gt.iloc[te_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.111, random_state=42)
tr_idx, va_idx = next(gss2.split(gt_tv, gt_tv['label'], groups=gt_tv['lesion_id']))
gt_train = gt_tv.iloc[tr_idx].reset_index(drop=True)
gt_val   = gt_tv.iloc[va_idx].reset_index(drop=True)

print(f"Train: {len(gt_train)} | Val: {len(gt_val)} | Test: {len(gt_test)}")

# Index images
all_jpgs   = glob.glob("/kaggle/input/**/*.jpg", recursive=True)
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in all_jpgs}
print(f"Images indexed: {len(img_lookup)}")
missing    = [r.image for _,r in gt.iterrows() if r.image not in img_lookup]
print(f"Missing: {len(missing)}")

In [ ]:
# Load frozen ViT backbone + extract ISIC 2019 embeddings
print("Loading ViT-B/16 SWAG...")
weights_vit = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
backbone    = vit_b_16(weights=weights_vit).to(DEVICE)
backbone.eval()
for p in backbone.parameters(): p.requires_grad = False
print(f"Frozen. Params: {sum(p.numel() for p in backbone.parameters()):,}")

transform = T.Compose([
    T.Resize(384, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(384), T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

class ISICDataset(Dataset):
    def __init__(self, df, lookup, tfm):
        self.df=df.reset_index(drop=True); self.lookup=lookup; self.tfm=tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r=self.df.iloc[i]; p=self.lookup.get(r.image)
        img=Image.open(p).convert('RGB') if p else Image.new('RGB',(384,384))
        return self.tfm(img), int(r.label), str(r.image)

@torch.no_grad()
def extract_isic(df, name, bs=64):
    ds=ISICDataset(df,img_lookup,transform)
    dl=DataLoader(ds,batch_size=bs,shuffle=False,num_workers=2,pin_memory=True)
    embs,lbls,ids=[],[],[]
    t0=time.time()
    for bi,(imgs,labels,imgids) in enumerate(dl):
        x=backbone._process_input(imgs.to(DEVICE))
        n=x.shape[0]
        x=torch.cat([backbone.class_token.expand(n,-1,-1),x],dim=1)
        cls=backbone.encoder(x)[:,0,:].cpu().float().numpy()
        embs.append(cls); lbls.append(labels.numpy()); ids.extend(imgids)
        done=min((bi+1)*bs,len(ds))
        if (bi+1)%20==0:
            ela=time.time()-t0
            print(f"  {name}: {done}/{len(ds)} | {ela:.0f}s | ETA {ela/done*(len(ds)-done):.0f}s")
    e=np.concatenate(embs); l=np.concatenate(lbls)
    print(f"  DONE {name}: {len(e)} embeddings | {time.time()-t0:.0f}s")
    return e,l,ids

print("\nExtracting ISIC 2019 embeddings...")
for name, df in [("train",gt_train),("val",gt_val),("test",gt_test)]:
    print(f"--- {name} ({len(df)} images) ---")
    e,l,ids = extract_isic(df, name)
    np.savez_compressed(f"{EMBED_DIR_B}/embeddings_{name}.npz",
                        embeddings=e, labels=l,
                        image_ids=np.array(ids,dtype=object))
    print(f"  Saved. Classes: {dict(sorted(Counter(l.tolist()).items()))}")

print("ISIC 2019 embeddings cached.")

In [ ]:
# Load ISIC 2019 embeddings + run FedAvg and PGA
def load_isic(name):
    d=np.load(f"{EMBED_DIR_B}/embeddings_{name}.npz",allow_pickle=True)
    return (torch.tensor(d['embeddings'],dtype=torch.float32),
            torch.tensor(d['labels'],    dtype=torch.long))

emb_tr19,lbl_tr19 = load_isic("train")
emb_va19,lbl_va19 = load_isic("val")
emb_te19,lbl_te19 = load_isic("test")

print(f"ISIC19 Train:{emb_tr19.shape} Val:{emb_va19.shape} Test:{emb_te19.shape}")

cnt19  = torch.bincount(lbl_tr19, minlength=NUM_CLASSES).float()
wt19   = (1.0/cnt19); wt19=(wt19/wt19.mean()).to(DEVICE)
print(f"Class weights: {wt19.cpu().numpy().round(3)}")

# BottleneckMLP with 8 output classes for ISIC 2019
class BottleneckMLP8(nn.Module):
    def __init__(self, ratio=4, n_classes=8):
        super().__init__()
        b=768//ratio
        self.net=nn.Sequential(
            nn.Linear(768,4*b),nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,b),  nn.GELU(),nn.LayerNorm(b),  nn.Dropout(0.2),
            nn.Linear(b,4*b),  nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,n_classes))
    def forward(self,x): return self.net(x)

In [ ]:
# Run FedAvg + PGA on ISIC 2019
def dirichlet_partition(labels,K,alpha,seed):
    rng=np.random.default_rng(seed); idx=[[] for _ in range(K)]
    nc=int(labels.max())+1
    for c in range(nc):
        ci=np.where(labels.numpy()==c)[0]; rng.shuffle(ci)
        props=rng.dirichlet(np.full(K,alpha))
        splits=(np.cumsum(props)*len(ci)).astype(int)[:-1]
        for k,chunk in enumerate(np.split(ci,splits)): idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

@torch.no_grad()
def evaluate19(head,embs,labels):
    head.eval(); p,t=[],[]
    for x,y in DataLoader(TensorDataset(embs,labels),batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu()); t.append(y)
    p=torch.cat(p).numpy(); t=torch.cat(t).numpy()
    f1s=[]
    for c in range(NUM_CLASSES):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9); f1s.append(2*pr*rc/(pr+rc+1e-9))
    return {"accuracy":float((p==t).mean()),"macro_f1":float(np.mean(f1s))}

def local_train19(head,embs,labels,lr=5e-3,epochs=5,bs=32):
    h=copy.deepcopy(head).to(DEVICE); h.train()
    opt=optim.Adam(h.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=wt19)
    for _ in range(epochs):
        for x,y in DataLoader(TensorDataset(embs.to(DEVICE),labels.to(DEVICE)),
                               batch_size=bs,shuffle=True):
            opt.zero_grad(); crit(h(x),y).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def fedavg(states,sizes):
    total=sum(sizes)
    return {k:sum((sizes[i]/total)*states[i][k].float() for i in range(len(sizes)))
            for k in states[0]}

print("="*55)
print("ISIC 2019: FedAvg vs PGA (K=4, 3 alpha, 5 seeds)")
print("="*55)
ALPHAS=[1000,1,0.1]; SEEDS=[0,1,2,3,4]; K=4
results_isic=[]

for method in ['fedavg','pga']:
    for alpha in ALPHAS:
        for seed in SEEDS:
            partition=dirichlet_partition(lbl_tr19,K,alpha,seed)
            c_embs  =[emb_tr19[idx] for idx in partition]
            c_labels=[lbl_tr19[idx] for idx in partition]
            sizes   =[len(idx) for idx in partition]
            if method=='pga':
                c_protos=[compute_prototypes(c_embs[k],c_labels[k])
                          for k in range(K)]
            head=BottleneckMLP8().to(DEVICE)
            best_val=-1; best_state=None
            for rnd in range(1,61):
                states=[local_train19(head,c_embs[k],c_labels[k])
                        for k in range(K)]
                if method=='pga':
                    head.load_state_dict(
                        prototype_guided_aggregation(states,sizes,c_protos))
                else:
                    head.load_state_dict(fedavg(states,sizes))
                vm=evaluate19(head,emb_va19.to(DEVICE),lbl_va19)
                if vm['macro_f1']>best_val:
                    best_val=vm['macro_f1']
                    best_state=copy.deepcopy(head.state_dict())
            head.load_state_dict(best_state)
            tm=evaluate19(head,emb_te19.to(DEVICE),lbl_te19)
            res={"method":method,"K":K,"alpha":alpha,"seed":seed,
                 "best_val_f1":best_val,"test":tm,"dataset":"ISIC2019"}
            results_isic.append(res)
            with open(os.path.join(RESULTS_DIR,
                      f"isic19_{method}_a{alpha}_s{seed}.json"),"w") as f:
                json.dump(res,f,indent=2)
        f1s=[r['test']['macro_f1'] for r in results_isic
             if r['method']==method and r['alpha']==alpha]
        print(f"  {method} alpha={alpha}: F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

with open(os.path.join(RESULTS_DIR,"isic2019_results.json"),"w") as f:
    json.dump(results_isic,f,indent=2)
print("\nISIC 2019 experiments complete.")
print("Save output as dataset: feduca-extra-b-results")